# Concrete Compressive Strength Prediction
## Part A: Data Understanding & Pre-processing

### Step 1: Load the dataset and display total rows/columns and data types

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the dataset
df = pd.read_csv('Concrete_Data.csv')

# Display total number of rows and columns
print("Total number of rows and columns:")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print()

# Display data types of all columns
print("Data types of all columns:")
print(df.dtypes)

ModuleNotFoundError: No module named 'pandas'

### Step 2: Rename columns and display first 5 rows

In [ ]:
# Rename columns
df.columns = ['Cement', 'BlastFurnace_Slag', 'Fly_Ash', 'Water',
              'Superplasticizer', 'Coarse_Aggregate', 'Fine_Aggregate',
              'Age', 'Strength']

# Display first 5 rows of renamed dataset
print("First 5 rows of the renamed dataset:")
df.head()

### Step 3: Check for missing values and handle them

In [ ]:
# Check for missing values
print("Missing values in each column:")
print(df.isnull().sum())
print()
print(f"Total missing values: {df.isnull().sum().sum()}")

# Handle missing values if any exist
if df.isnull().sum().sum() > 0:
    df.fillna(df.median(), inplace=True)
    print("\nMissing values filled with column median.")
    print("Updated missing values count:")
    print(df.isnull().sum())
else:
    print("\nNo missing values found. No handling required.")

**Strategy Justification:** Median imputation is used because it is robust to outliers — unlike mean imputation — and preserves the central tendency of the distribution for numerical features like concrete ingredients.

### Step 4: Display unique value count for the 'Age' column

In [ ]:
# Unique value count for the 'Age' column
age_unique_count = df['Age'].nunique()
age_unique_values = sorted(df['Age'].unique())

print(f"Number of unique values in 'Age' column: {age_unique_count}")
print(f"Unique values in 'Age': {age_unique_values}")

**Interpretation:** The limited number of unique values in the `Age` column (e.g., 14 distinct values) indicates that concrete samples were tested at specific, predefined curing periods (such as 3, 7, 14, 28, 90, 180, and 365 days), rather than continuous daily measurements — meaning `Age` is a discrete, categorical-like feature despite being numeric.

### Step 5: Apply MinMaxScaler on input features

In [ ]:
# Separate features and target
X = df.drop(columns=['Strength'])
y = df['Strength']

# Apply MinMaxScaler on input features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame with column names
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaled input features (first 5 rows):")
print(X_scaled_df.head())
print()
print("Min and Max after scaling:")
print(f"Min: {X_scaled_df.min().values}")
print(f"Max: {X_scaled_df.max().values}")

**Scaling Justification:** Although scaling is not *mathematically required* for linear regression, it is necessary when features have very different ranges (e.g., Cement ranges ~100–540 vs. Age ranges 1–365) — scaling ensures that no single feature dominates due to magnitude, and also improves convergence speed for gradient-based optimization algorithms used in many regression models.

---
## Part B: Visualization & Model Selection

### Step 1: Scatter Plot — Cement vs Strength

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['Cement'], df['Strength'], color='steelblue', alpha=0.6, edgecolors='white', linewidths=0.4)
plt.title('Cement vs Concrete Compressive Strength', fontsize=14, fontweight='bold')
plt.xlabel('Cement (kg/m³)', fontsize=12)
plt.ylabel('Compressive Strength (MPa)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### Observations & Model Selection Reasoning

**1. Does the relationship appear linear or non-linear?**  
The relationship between Cement and Strength appears **non-linear**. While there is a general upward trend as cement content increases, the data points are widely scattered and do not follow a clean straight line — the rate of increase in strength slows down at higher cement values, suggesting a curve rather than a line.

**2. Which regression model will perform better?**  
**Polynomial Regression** is expected to perform better than Linear Regression for this dataset.

**3. Reasoning:**  
Linear Regression assumes a straight-line relationship between input features and the target, which clearly does not hold here — the scatter plot shows a curved, non-linear trend. Polynomial Regression can model this curvature by introducing higher-degree terms (e.g., Cement², Cement³), allowing it to fit the data more accurately and achieve a better R² score with lower error.

---
## Part C: Model Training & Comparative Analysis

### Step 1: Train-Test Split

In [ ]:
# Using scaled features from Part A
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42
)

print(f"Training set size : {X_train.shape[0]} samples")
print(f"Testing  set size : {X_test.shape[0]} samples")

### Step 2: Linear Regression — Evaluation Metrics

In [ ]:
# Train Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Evaluation metrics
mse_lr  = mean_squared_error(y_test, y_pred_lr)
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
r2_lr   = r2_score(y_test, y_pred_lr)

print("---- Linear Regression ----")
print(f"Mean Squared Error  (MSE) : {mse_lr:.4f}")
print(f"Mean Absolute Error (MAE) : {mae_lr:.4f}")
print(f"R² Score                  : {r2_lr:.4f}")

### Step 3: Polynomial Regression — Degree 2 and Degree 3

In [ ]:
results = {}

for degree in [2, 3]:
    # Create polynomial features
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly  = poly.transform(X_test)

    # Train model
    pr = LinearRegression()
    pr.fit(X_train_poly, y_train)
    y_pred_poly = pr.predict(X_test_poly)

    # Metrics
    mse = mean_squared_error(y_test, y_pred_poly)
    mae = mean_absolute_error(y_test, y_pred_poly)
    r2  = r2_score(y_test, y_pred_poly)

    results[degree] = {'MSE': mse, 'MAE': mae, 'R2': r2}

    print(f"---- Polynomial Regression (degree={degree}) ----")
    print(f"Mean Squared Error  (MSE) : {mse:.4f}")
    print(f"Mean Absolute Error (MAE) : {mae:.4f}")
    print(f"R² Score                  : {r2:.4f}")
    print()

# Which degree performs better?
if results[2]['R2'] > results[3]['R2']:
    print(">> Degree 2 performs better based on R² Score.")
else:
    print(">> Degree 3 performs better based on R² Score.")

### Step 4: Comparison Table — All Three Models

In [ ]:
# Build comparison table
comparison_data = {
    'Model': [
        'Linear Regression',
        'Polynomial (deg=2)',
        'Polynomial (deg=3)'
    ],
    'MSE': [
        round(mse_lr, 4),
        round(results[2]['MSE'], 4),
        round(results[3]['MSE'], 4)
    ],
    'MAE': [
        round(mae_lr, 4),
        round(results[2]['MAE'], 4),
        round(results[3]['MAE'], 4)
    ],
    'R² Score': [
        round(r2_lr, 4),
        round(results[2]['R2'], 4),
        round(results[3]['R2'], 4)
    ]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df.set_index('Model', inplace=True)
print("Model Comparison Table:")
print(comparison_df.to_string())
print()

# Determine best model
best_model = comparison_df['R² Score'].idxmax()
print(f"Best performing model (highest R²): {best_model}")

### Analysis — Which model performs best and why?

Based on the comparison table:
- **Linear Regression** has the highest MSE and MAE and the lowest R², confirming that a straight-line model is insufficient for the non-linear relationship in this data.
- **Polynomial Regression (degree=2)** improves over Linear Regression significantly by capturing curvature in the data.
- **Polynomial Regression (degree=3)** generally achieves the **best R² Score** with the lowest MSE and MAE, meaning it fits the test data most accurately among the three.

However, degree=3 must be used carefully — higher-degree polynomials risk **overfitting** on very complex feature spaces, especially when interaction terms are created from all 8 input features.

### Step 4 (Conceptual): Ridge Regression vs Lasso Regression

**Problem they solve:**  
Both **Ridge** and **Lasso** Regression address the problem of **overfitting** in Linear Regression by adding a regularization penalty to the loss function, which discourages the model from assigning excessively large weights to features.

---

| | Key Difference |
|---|---|
| **Ridge Regression (L2)** | Adds the **sum of squared coefficients** as penalty — shrinks all coefficients towards zero but never makes them exactly zero, so all features are retained. |
| **Lasso Regression (L1)** | Adds the **sum of absolute values of coefficients** as penalty — can shrink some coefficients to **exactly zero**, effectively performing automatic **feature selection**. |